# Advanced Agent Frameworks
A comprehensive guide to production-grade agent frameworks: MetaGPT, OpenAI Agents SDK, AutoGPT, BabyAGI, MemGPT/Letta, SuperAGI, Phidata/Agno, Composio, and more.

## Table of Contents
1. MetaGPT Software Company as Multi-Agent System
2. OpenAI Agents SDK (formerly Swarm)
3. AutoGPT Original Autonomous Agent
4. BabyAGI Minimal Task-Driven Agent
5. MemGPT / Letta OS-Inspired Memory Management
6. SuperAGI Production Agent Framework
7. Phidata / Agno Workflow-Based Agents
8. Composio Tool Integration Platform
9. Agent Comparison Matrix
10. Agent Failure Modes
11. Agent Evaluation Frameworks
12. Computer Use Agents
13. Voice Agents
14. Additional Learning Resources

## 1. MetaGPT Software Company as Multi-Agent System

MetaGPT models a software company as a **multi-agent system** where each agent plays a specialized role. The key insight is that structured outputs (PRDs, system designs, code reviews) act as **standardized interfaces** between roles just like documents in a real company.

### Role Hierarchy

```
User Story / Idea
      │
      ▼
┌─────────────┐
│Product Mgr  │──► PRD (Product Requirements Doc)
└─────────────┘
      │
      ▼
┌─────────────┐
│  Architect  │──► System Design (APIs, DB Schema, Classes)
└─────────────┘
      │
      ▼
┌─────────────┐
│Project Mgr  │──► Task breakdown, Sprint planning
└─────────────┘
      │
      ▼
┌─────────────┐
│  Engineer   │──► Code implementation
└─────────────┘
      │
      ▼
┌─────────────┐
│ QA Engineer │──► Test cases, Bug reports
└─────────────┘
```

### Core Concepts

| Concept | Description |
|---------|-------------|
| **Role** | Agent with a specific persona, goal, and set of Actions |
| **Action** | Atomic unit of work (WritePRD, WriteCode, RunTests) |
| **Message Bus** | Shared pub/sub system; roles subscribe to message types |
| **Memory** | Role-local storage of messages and outputs |
| **Environment** | Shared workspace where roles collaborate |

### Message Flow

Each role **publishes** structured outputs and **subscribes** to outputs from upstream roles:

$$\text{Role}_i \xrightarrow{\text{Action}} \text{Message}(\text{structured output}) \rightarrow \text{Bus} \rightarrow \text{Role}_{i+1}$$

The message bus ensures **loose coupling** roles don't call each other directly, they react to messages.

In [1]:
# MetaGPT-style Mini Software Development Team
# Simulates the MetaGPT architecture without requiring the full library

import json
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
from enum import Enum

# --- Message Bus ---
class MessageBus:
    """Simple pub/sub message bus for inter-agent communication."""
    def __init__(self):
        self.messages: List[Dict] = []
        self.subscribers: Dict[str, List] = {}
    
    def subscribe(self, message_type: str, handler):
        self.subscribers.setdefault(message_type, []).append(handler)
    
    def publish(self, message_type: str, content: Any, sender: str):
        msg = {"type": message_type, "content": content, "sender": sender}
        self.messages.append(msg)
        print(f"\n📨 [{sender}] → [{message_type}]")
        for handler in self.subscribers.get(message_type, []):
            handler(msg)

# --- Base Role ---
@dataclass
class Role:
    name: str
    profile: str
    bus: MessageBus
    memory: List[Dict] = field(default_factory=list)
    
    def remember(self, msg: Dict):
        self.memory.append(msg)
    
    def act(self, context: str) -> str:
        raise NotImplementedError

# --- Concrete Roles ---
class ProductManager(Role):
    def act(self, idea: str) -> str:
        prd = {
            "title": f"PRD: {idea[:50]}",
            "goals": ["Build core feature", "Ensure usability", "MVP in 2 weeks"],
            "user_stories": [
                f"As a user, I want to {idea.lower()} so that I can save time.",
                "As a user, I want clear error messages.",
            ],
            "requirements": ["REQ-001: Core functionality", "REQ-002: Error handling", "REQ-003: Logging"],
            "out_of_scope": ["Mobile app", "i18n"],
        }
        self.bus.publish("PRD", prd, self.name)
        return prd

class Architect(Role):
    def on_prd(self, msg: Dict):
        prd = msg["content"]
        design = {
            "components": ["API Layer", "Business Logic", "Data Layer", "Utils"],
            "api_endpoints": ["POST /execute", "GET /status/{id}", "DELETE /cancel/{id}"],
            "data_models": {
                "Task": {"id": "str", "goal": "str", "status": "enum", "result": "Optional[str]"},
                "Result": {"task_id": "str", "output": "str", "error": "Optional[str]"},
            },
            "tech_stack": {"language": "Python 3.10", "framework": "FastAPI", "db": "SQLite"},
            "based_on_requirements": prd["requirements"],
        }
        self.remember(msg)
        self.bus.publish("SystemDesign", design, self.name)

class Engineer(Role):
    def on_design(self, msg: Dict):
        design = msg["content"]
        code_stubs = {}
        for component in design["components"]:
            class_name = component.replace(" ", "")
            code_stubs[component] = f'''class {class_name}:
    """Auto-generated stub for {component}"""
    def __init__(self):
        self._initialized = True
    
    def execute(self, *args, **kwargs):
        # TODO: implement {component} logic
        raise NotImplementedError
'''
        self.remember(msg)
        self.bus.publish("Code", {"stubs": code_stubs, "files": list(code_stubs.keys())}, self.name)
        print(f"  Generated {len(code_stubs)} code stubs")

class QAEngineer(Role):
    def on_code(self, msg: Dict):
        code = msg["content"]
        tests = []
        for filename in code["files"]:
            tests.append({
                "file": f"test_{filename.lower().replace(' ', '_')}.py",
                "cases": [
                    f"test_{filename.lower().replace(' ', '_')}_init",
                    f"test_{filename.lower().replace(' ', '_')}_execute_raises",
                    f"test_{filename.lower().replace(' ', '_')}_edge_cases",
                ]
            })
        self.remember(msg)
        self.bus.publish("TestReport", {"tests": tests, "coverage_target": "80%"}, self.name)
        print(f"  Generated {len(tests)} test files")

# --- Assemble the Team ---
def run_software_team(idea: str):
    print(f"{'='*60}")
    print(f"💡 Idea: {idea}")
    print(f"{'='*60}")
    
    bus = MessageBus()
    
    # Instantiate roles
    pm = ProductManager("Alice (PM)", "Product Manager", bus)
    arch = Architect("Bob (Architect)", "Software Architect", bus)
    eng = Engineer("Carol (Engineer)", "Senior Engineer", bus)
    qa = QAEngineer("Dave (QA)", "QA Engineer", bus)
    
    # Wire subscriptions
    bus.subscribe("PRD", arch.on_prd)
    bus.subscribe("SystemDesign", eng.on_design)
    bus.subscribe("Code", qa.on_code)
    
    # Kick off the pipeline
    pm.act(idea)
    
    print(f"\n{'='*60}")
    print(f"✅ Pipeline complete. Total messages: {len(bus.messages)}")
    print(f"{'='*60}")
    
    return bus.messages

# Run it
messages = run_software_team("build a CLI tool that summarizes GitHub pull requests using an LLM")

# Show final outputs
print("\n--- Final Artifacts ---")
for msg in messages:
    print(f"\n[{msg['type']}] from {msg['sender']}:")
    if isinstance(msg['content'], dict):
        for k, v in msg['content'].items():
            if k != 'stubs':  # skip verbose code
                print(f"  {k}: {v}")

💡 Idea: build a CLI tool that summarizes GitHub pull requests using an LLM

📨 [Alice (PM)] → [PRD]

📨 [Bob (Architect)] → [SystemDesign]

📨 [Carol (Engineer)] → [Code]

📨 [Dave (QA)] → [TestReport]
  Generated 4 test files
  Generated 4 code stubs

✅ Pipeline complete. Total messages: 4

--- Final Artifacts ---

[PRD] from Alice (PM):
  title: PRD: build a CLI tool that summarizes GitHub pull reque
  goals: ['Build core feature', 'Ensure usability', 'MVP in 2 weeks']
  user_stories: ['As a user, I want to build a cli tool that summarizes github pull requests using an llm so that I can save time.', 'As a user, I want clear error messages.']
  requirements: ['REQ-001: Core functionality', 'REQ-002: Error handling', 'REQ-003: Logging']
  out_of_scope: ['Mobile app', 'i18n']

[SystemDesign] from Bob (Architect):
  components: ['API Layer', 'Business Logic', 'Data Layer', 'Utils']
  api_endpoints: ['POST /execute', 'GET /status/{id}', 'DELETE /cancel/{id}']
  data_models: {'Task': {'id': 's

## 2. OpenAI Agents SDK (formerly Swarm)

The **OpenAI Agents SDK** (evolved from the experimental Swarm library) provides a production-ready framework for multi-agent orchestration. Key primitives:

| Primitive | Description |
|-----------|-------------|
| **Agent** | An LLM with instructions, tools, and optional handoff targets |
| **Handoff** | Transfer control to another agent (modeled as a tool call) |
| **Guardrail** | Input/output validation that can trip-wire and redirect |
| **Runner** | Orchestrates the agent loop with streaming support |
| **Trace** | Full audit log of all LLM calls, tool uses, handoffs |

### Agent Loop

```
User Input
    │
    ▼
┌──────────┐    tool call     ┌──────────────┐
│  Agent A │ ──────────────► │  Tool / API  │
│          │ ◄────────────── │   Result     │
└──────────┘    result        └──────────────┘
    │
    │ handoff(AgentB)
    ▼
┌──────────┐
│  Agent B │ ──► continues conversation
└──────────┘
```

### Handoffs vs. Tool Calls

A **handoff** is a special tool call that transfers the entire conversation context to a new agent. The new agent takes over as the primary responder:

$$\text{Handoff} = \text{transfer\_to\_agent}(\text{agent\_name}, \text{context}) \rightarrow \text{New Agent takes over}$$

### Guardrails

Guardrails run **in parallel** with the agent they monitor inputs/outputs and can raise `TripwireTriggered` to halt execution:

```python
# Guardrail pattern
async def safety_guardrail(ctx, agent, input):
    result = await Runner.run(guardrail_agent, input)
    if result.final_output.is_unsafe:
        raise InputGuardrailTripwireTriggered(result)
    return GuardrailFunctionOutput(output_info=result, tripwire_triggered=False)
```

In [2]:
# OpenAI Agents SDK Multi-Agent Handoff Example
# Demonstrates Agent, handoffs, tools, and tracing patterns
# (Uses mock LLM calls so it runs without an API key)

import asyncio
import nest_asyncio; nest_asyncio.apply()  # allow asyncio.run inside Jupyter loop
import json
from dataclasses import dataclass, field
from typing import List, Dict, Any, Callable, Optional
from enum import Enum

# --- Mock Agent SDK (mirrors real openai-agents API) ---

@dataclass
class Tool:
    name: str
    description: str
    func: Callable
    
    def to_schema(self):
        return {"type": "function", "function": {"name": self.name, "description": self.description}}

@dataclass  
class Agent:
    name: str
    instructions: str
    tools: List[Tool] = field(default_factory=list)
    handoffs: List['Agent'] = field(default_factory=list)
    model: str = "gpt-4o"
    
    def as_handoff_tool(self) -> Tool:
        """Make this agent available as a handoff target."""
        return Tool(
            name=f"transfer_to_{self.name.lower().replace(' ', '_')}",
            description=f"Transfer the conversation to {self.name}",
            func=lambda **kwargs: f"[Handoff to {self.name}]"
        )

class RunResult:
    def __init__(self, final_output: str, trace: List[Dict]):
        self.final_output = final_output
        self.trace = trace

class Runner:
    """Simplified agent runner with tracing."""
    
    @staticmethod
    async def run(agent: Agent, user_input: str, context: Dict = None) -> RunResult:
        trace = []
        context = context or {}
        
        # Simulate the agent loop
        trace.append({
            "type": "agent_start",
            "agent": agent.name,
            "input": user_input,
        })
        
        # Mock: detect if handoff is needed based on keywords
        response, handoff_target = Runner._mock_llm_call(agent, user_input, context)
        
        trace.append({
            "type": "llm_response",
            "agent": agent.name,
            "response": response,
        })
        
        if handoff_target:
            # Find the target agent
            target = next((h for h in agent.handoffs if h.name == handoff_target), None)
            if target:
                trace.append({
                    "type": "handoff",
                    "from": agent.name,
                    "to": target.name,
                    "reason": response,
                })
                print(f"  🔄 Handoff: {agent.name} → {target.name}")
                # Recurse into the new agent
                sub_result = await Runner.run(target, user_input, context)
                trace.extend(sub_result.trace)
                return RunResult(sub_result.final_output, trace)
        
        # Execute any tool calls
        for tool_call in Runner._mock_tool_calls(agent, user_input):
            tool = next((t for t in agent.tools if t.name == tool_call["name"]), None)
            if tool:
                result = tool.func(**tool_call.get("args", {}))
                trace.append({
                    "type": "tool_call",
                    "tool": tool_call["name"],
                    "result": result,
                })
                response += f"\n[Tool {tool_call['name']} returned: {result}]"
        
        trace.append({"type": "agent_end", "agent": agent.name, "final_output": response})
        return RunResult(response, trace)
    
    @staticmethod
    def _mock_llm_call(agent: Agent, user_input: str, context: Dict):
        """Mock LLM returns (response, handoff_target_name_or_None)."""
        user_lower = user_input.lower()
        
        if agent.name == "Triage Agent":
            if any(w in user_lower for w in ["bill", "charge", "invoice", "payment", "refund"]):
                return "This looks like a billing question. Let me transfer you.", "Billing Agent"
            elif any(w in user_lower for w in ["bug", "error", "crash", "broken", "issue", "technical"]):
                return "This is a technical issue. Routing to tech support.", "Technical Support Agent"
            else:
                return f"Hello! I'm the triage agent. Your query: '{user_input}'. How can I help?", None
        
        elif agent.name == "Billing Agent":
            return f"[Billing] I can help with your billing concern: '{user_input}'. Your account shows no outstanding issues. Refund will process in 3-5 days.", None
        
        elif agent.name == "Technical Support Agent":
            return f"[Tech Support] I see you're having a technical issue: '{user_input}'. Please try: 1) Clear cache, 2) Update to latest version, 3) Restart the service.", None
        
        return f"[{agent.name}] Processed: {user_input}", None
    
    @staticmethod
    def _mock_tool_calls(agent: Agent, user_input: str):
        """Mock tool call detection."""
        calls = []
        if "search" in user_input.lower() and any(t.name == "web_search" for t in agent.tools):
            calls.append({"name": "web_search", "args": {"query": user_input}})
        return calls

# --- Define Tools ---
def web_search(query: str) -> str:
    return f"Search results for '{query}': [Result 1] [Result 2] [Result 3]"

def get_account_info(account_id: str = "demo") -> str:
    return json.dumps({"account_id": account_id, "status": "active", "balance": "$0.00"})

search_tool = Tool("web_search", "Search the web for information", web_search)
account_tool = Tool("get_account_info", "Look up customer account details", get_account_info)

# --- Define Agents ---
billing_agent = Agent(
    name="Billing Agent",
    instructions="You handle billing, payments, refunds, and invoice questions. Be concise and helpful.",
    tools=[account_tool],
)

tech_agent = Agent(
    name="Technical Support Agent", 
    instructions="You handle technical issues, bugs, and errors. Provide step-by-step solutions.",
    tools=[search_tool],
)

triage_agent = Agent(
    name="Triage Agent",
    instructions="You are the first point of contact. Route to billing or tech support as appropriate.",
    tools=[search_tool],
    handoffs=[billing_agent, tech_agent],
)

# --- Run Examples ---
async def demo_multi_agent():
    test_queries = [
        "I was charged twice for my subscription last month",
        "The application keeps crashing when I click the export button",
        "What are your business hours?",
    ]
    
    for query in test_queries:
        print(f"\n{'─'*60}")
        print(f"👤 User: {query}")
        result = await Runner.run(triage_agent, query)
        print(f"🤖 Final: {result.final_output}")
        print(f"📊 Trace steps: {len(result.trace)}")
        
        # Show trace summary
        for step in result.trace:
            if step["type"] == "handoff":
                print(f"   ↳ Handoff: {step['from']} → {step['to']}")

asyncio.run(demo_multi_agent())


────────────────────────────────────────────────────────────
👤 User: I was charged twice for my subscription last month
  🔄 Handoff: Triage Agent → Billing Agent
🤖 Final: [Billing] I can help with your billing concern: 'I was charged twice for my subscription last month'. Your account shows no outstanding issues. Refund will process in 3-5 days.
📊 Trace steps: 6
   ↳ Handoff: Triage Agent → Billing Agent

────────────────────────────────────────────────────────────
👤 User: The application keeps crashing when I click the export button
  🔄 Handoff: Triage Agent → Technical Support Agent
🤖 Final: [Tech Support] I see you're having a technical issue: 'The application keeps crashing when I click the export button'. Please try: 1) Clear cache, 2) Update to latest version, 3) Restart the service.
📊 Trace steps: 6
   ↳ Handoff: Triage Agent → Technical Support Agent

────────────────────────────────────────────────────────────
👤 User: What are your business hours?
🤖 Final: Hello! I'm the tria

## 3. AutoGPT Original Autonomous Agent

AutoGPT (2023) was one of the first viral autonomous agents it took a **natural language goal** and tried to achieve it without human intervention.

### Architecture

```
┌─────────────────────────────────────────────────────┐
│                    AutoGPT Loop                      │
│                                                       │
│  ┌─────────┐    ┌─────────┐    ┌─────────┐          │
│  │  THINK  │───►│  PLAN   │───►│   ACT   │          │
│  │         │    │         │    │         │          │
│  │ Reason  │    │ Break   │    │ Execute │          │
│  │ about   │    │ into    │    │ command │          │
│  │ goal    │    │ steps   │    │ (tool)  │          │
│  └─────────┘    └─────────┘    └────┬────┘          │
│       ▲                              │               │
│       │         ┌─────────┐          │               │
│       └─────────│ REFLECT │◄─────────┘               │
│                 │         │                          │
│                 │ Update  │                          │
│                 │ memory  │                          │
│                 └─────────┘                          │
└─────────────────────────────────────────────────────┘
```

### Memory Architecture

AutoGPT uses a **two-tier memory** system:

| Memory Type | Implementation | Capacity | Use |
|-------------|---------------|----------|-----|
| Short-term | LLM context window | ~4K-8K tokens | Recent actions, current plan |
| Long-term | VectorDB (Pinecone/Redis) | Unlimited | Past results, learned facts |

### Commands / Tools

AutoGPT had a fixed set of commands:
- `google` web search
- `browse_website` fetch and parse URL
- `write_to_file` / `read_file` file I/O
- `execute_python_file` run code
- `task_complete` signal goal achieved

### Why AutoGPT Struggled with Long-Horizon Tasks

1. **Context overflow**: After many steps, early context is lost or truncated
2. **Error accumulation**: Small mistakes compound no backtracking
3. **Goal drift**: LLM re-interprets the goal each step, causing drift
4. **Infinite loops**: No loop detection; agents would repeat actions
5. **Hallucinated tool use**: Agents would "use" tools that didn't exist or return fabricated results

$$\text{Success Rate} \propto \frac{1}{\text{Horizon Length}^2}$$

AutoGPT demonstrated the **concept** of autonomous agents but revealed fundamental limitations of purely LLM-driven long-horizon planning.

## 4. BabyAGI Minimal Task-Driven Agent

BabyAGI (Nakajima, 2023) distilled AutoGPT's ideas into a clean **three-loop architecture**:

```
┌──────────────────────────────────────────────┐
│                 BabyAGI Loop                  │
│                                               │
│  Task Queue (prioritized)                     │
│  ┌─────────────────────────────────┐         │
│  │ 1. Research X    (priority: 90) │         │
│  │ 2. Summarize Y   (priority: 70) │         │
│  │ 3. Write report  (priority: 50) │ ◄──┐   │
│  └─────────────────────────────────┘    │   │
│          │                              │   │
│          ▼                              │   │
│  ┌───────────────┐                      │   │
│  │ 1. EXECUTION  │ ── LLM ──► result   │   │
│  │    AGENT      │                      │   │
│  └───────┬───────┘                      │   │
│          │ result                        │   │
│          ▼                              │   │
│  ┌───────────────┐                      │   │
│  │ 2. CREATION   │ ── LLM ──► new tasks─┘   │
│  │    AGENT      │                          │
│  └───────┬───────┘                          │
│          │ task list                         │
│          ▼                                  │
│  ┌───────────────┐                          │
│  │3. PRIORITIZE  │ ── LLM ──► reordered    │
│  │   AGENT       │           task list      │
│  └───────────────┘                          │
└──────────────────────────────────────────────┘
```

The VectorDB stores results, enabling semantic lookup: *"Have I already researched this?"*

In [3]:
# BabyAGI Implementation ~50 lines of core logic
# Faithful to the original architecture (Nakajima 2023)

import json
from collections import deque
from typing import List, Dict, Optional

# --- Mock LLM (replace with real OpenAI/Anthropic call in production) ---
def llm(prompt: str, max_tokens: int = 200) -> str:
    """Mock LLM in production: openai.chat.completions.create(...)"""
    if "create tasks" in prompt.lower():
        # Simulate task creation
        prev = prompt.split("PREVIOUS TASK:")[-1][:50] if "PREVIOUS TASK:" in prompt else ""
        return json.dumps([
            f"Research the key aspects of: {prev[:30]}",
            f"Analyze implications of findings from: {prev[:30]}",
            f"Draft a summary report based on research",
        ])
    elif "prioritize" in prompt.lower():
        tasks = prompt.split("\n")
        tasks = [t for t in tasks if t.strip() and not t.startswith("Prioritize")]
        return "\n".join(f"{i+1}. {t.strip()}" for i, t in enumerate(tasks[:5]))
    else:
        # Simulate task execution
        task = prompt.split("TASK:")[-1].strip()[:60] if "TASK:" in prompt else prompt[:60]
        return f"Completed analysis of '{task}'. Key findings: [1] Initial research shows promising results. [2] Further investigation recommended. [3] Data suggests strong correlation."

# --- Mock VectorDB ---
class VectorMemory:
    """Simplified vector memory (use ChromaDB/Pinecone in production)."""
    def __init__(self):
        self.store: List[Dict] = []
    
    def add(self, task: str, result: str):
        self.store.append({"task": task, "result": result})
    
    def query(self, query: str, n: int = 3) -> List[str]:
        # Simple keyword match (real impl uses embeddings)
        scored = [(s, sum(w in s["task"].lower() for w in query.lower().split())) 
                  for s in self.store]
        return [s["result"] for s, _ in sorted(scored, key=lambda x: -x[1])[:n] if s]

# --- BabyAGI Core ---
class BabyAGI:
    def __init__(self, objective: str, initial_task: str, max_iterations: int = 5):
        self.objective = objective
        self.task_list = deque()
        self.memory = VectorMemory()
        self.task_id_counter = 1
        self.max_iterations = max_iterations
        
        # Seed the queue
        self.task_list.append({"id": self.task_id_counter, "task": initial_task})
        self.task_id_counter += 1
    
    def execution_agent(self, task: str) -> str:
        """Execute a task using LLM + memory context."""
        context = self.memory.query(task, n=3)
        context_str = "\n".join(context) if context else "No prior context."
        prompt = (
            f"OBJECTIVE: {self.objective}\n"
            f"TASK: {task}\n"
            f"CONTEXT (from memory):\n{context_str}\n"
            f"Complete the task and provide a detailed response:"
        )
        return llm(prompt)
    
    def task_creation_agent(self, result: str, task: str) -> List[str]:
        """Create new tasks based on completed task result."""
        prompt = (
            f"OBJECTIVE: {self.objective}\n"
            f"PREVIOUS TASK: {task}\n"
            f"RESULT: {result[:200]}\n"
            f"Based on the result, create new tasks as a JSON list of strings.\n"
            f"create tasks:"
        )
        try:
            raw = llm(prompt)
            new_tasks = json.loads(raw)
            return new_tasks if isinstance(new_tasks, list) else []
        except json.JSONDecodeError:
            return []
    
    def prioritization_agent(self) -> None:
        """Re-prioritize the task list."""
        task_names = [t["task"] for t in self.task_list]
        prompt = f"Prioritize these tasks for objective '{self.objective}':\n" + "\n".join(task_names)
        result = llm(prompt)
        
        # Rebuild task list from prioritized output
        lines = [l.strip() for l in result.split("\n") if l.strip()]
        self.task_list.clear()
        for line in lines:
            # Strip leading "1. ", "2. " etc.
            task_name = line.lstrip("0123456789. ").strip()
            if task_name:
                self.task_list.append({"id": self.task_id_counter, "task": task_name})
                self.task_id_counter += 1
    
    def run(self):
        print(f"\n{'='*60}")
        print(f"🎯 OBJECTIVE: {self.objective}")
        print(f"{'='*60}")
        
        for iteration in range(1, self.max_iterations + 1):
            if not self.task_list:
                print("\n✅ All tasks completed!")
                break
            
            # 1. Pull next task
            current = self.task_list.popleft()
            print(f"\n[Iteration {iteration}] 📋 Task #{current['id']}: {current['task']}")
            print(f"  Queue depth: {len(self.task_list)} remaining tasks")
            
            # 2. Execute
            result = self.execution_agent(current["task"])
            print(f"  ✔ Result: {result[:100]}...")
            
            # 3. Store in memory
            self.memory.add(current["task"], result)
            
            # 4. Create new tasks
            new_tasks = self.task_creation_agent(result, current["task"])
            for task_name in new_tasks[:2]:  # Limit growth
                self.task_list.append({"id": self.task_id_counter, "task": task_name})
                self.task_id_counter += 1
            if new_tasks:
                print(f"  ➕ Created {min(len(new_tasks), 2)} new tasks")
            
            # 5. Reprioritize
            if self.task_list:
                self.prioritization_agent()
        
        print(f"\n{'='*60}")
        print(f"📊 Memory entries: {len(self.memory.store)}")
        print(f"📝 Tasks processed: {iteration}")
        print(f"{'='*60}")
        return self.memory.store

# Run BabyAGI
agent = BabyAGI(
    objective="Research the latest advances in multimodal AI and prepare a summary report",
    initial_task="Search for recent papers on multimodal AI published in 2024",
    max_iterations=4,
)
results = agent.run()


🎯 OBJECTIVE: Research the latest advances in multimodal AI and prepare a summary report

[Iteration 1] 📋 Task #1: Search for recent papers on multimodal AI published in 2024
  Queue depth: 0 remaining tasks
  ✔ Result: Completed analysis of 'Search for recent papers on multimodal AI published in 2024
'. Key findings: ...
  ➕ Created 2 new tasks

[Iteration 2] 📋 Task #4: Research the key aspects of:  Search for recent papers on m
  Queue depth: 1 remaining tasks
  ✔ Result: Completed analysis of 'Research the key aspects of:  Search for recent papers on m
'. Key findings: ...
  ➕ Created 2 new tasks

[Iteration 3] 📋 Task #8: Analyze implications of findings from:  Search for recent papers on m
  Queue depth: 2 remaining tasks
  ✔ Result: Completed analysis of 'Analyze implications of findings from:  Search for recent pa'. Key findings: ...
  ➕ Created 2 new tasks

[Iteration 4] 📋 Task #13: Research the key aspects of:  Research the key aspects of:
  Queue depth: 3 remaining tasks
  ✔ R

## 5. MemGPT / Letta OS-Inspired Memory Management

**MemGPT** (now rebranded as **Letta**) applies operating system concepts to LLM memory management. The core insight: an LLM's context window is like RAM limited but fast while external storage is like disk unlimited but requires explicit I/O.

### Memory Hierarchy

```
┌─────────────────────────────────────────────────────┐
│                   MemGPT Agent                       │
│                                                       │
│  ┌─────────────────────────────────────────────┐    │
│  │           MAIN CONTEXT (RAM)                 │    │
│  │  ┌──────────┐ ┌──────────┐ ┌─────────────┐  │    │
│  │  │  System  │ │  Human   │ │   Persona   │  │    │
│  │  │ Prompt   │ │  Block   │ │    Block    │  │    │
│  │  └──────────┘ └──────────┘ └─────────────┘  │    │
│  │                                               │    │
│  │  ┌─────────────────────────────────────┐    │    │
│  │  │      Conversation History           │    │    │
│  │  │      (last N messages)              │    │    │
│  │  └─────────────────────────────────────┘    │    │
│  └──────────────────────┬──────────────────────┘    │
│                          │ memory functions           │
│           ┌──────────────┼──────────────┐            │
│           ▼              ▼              ▼            │
│  ┌──────────────┐ ┌──────────────┐ ┌──────────┐    │
│  │   ARCHIVAL   │ │    RECALL    │ │  WORKING │    │
│  │   MEMORY     │ │   MEMORY     │ │  MEMORY  │    │
│  │  (VectorDB)  │ │  (Conv Hist) │ │  (temp)  │    │
│  │  unlimited   │ │  searchable  │ │  scratch │    │
│  └──────────────┘ └──────────────┘ └──────────┘    │
└─────────────────────────────────────────────────────┘
```

### Memory Functions (LLM-callable tools)

| Function | Action |
|----------|--------|
| `core_memory_append(name, content)` | Add to a named memory block |
| `core_memory_replace(name, old, new)` | Edit an existing memory block |
| `archival_memory_insert(content)` | Write to VectorDB |
| `archival_memory_search(query)` | Retrieve from VectorDB |
| `conversation_search(query)` | Search past conversations |
| `send_message(content)` | Reply to user (only real output) |

The agent can **only communicate with the user via `send_message`** all other function calls are internal. This creates a clear boundary between thinking (internal) and speaking (external).

### Virtual Context Management

When context approaches the limit, MemGPT automatically:
1. Summarizes old conversation turns
2. Moves summaries to archival memory
3. Frees up context for new messages

$$\text{Effective Memory} = \underbrace{\text{Context Window}}_{\text{active}} + \underbrace{\text{VectorDB}}_{\text{archival}} \rightarrow \infty$$

In [4]:
# MemGPT-style Agent with Memory Tool Calls
# Demonstrates virtual context management and memory functions

import json
import time
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
from collections import deque

# --- Memory Storage ---
@dataclass
class CoreMemoryBlock:
    """Named, editable memory block in the main context."""
    name: str
    content: str
    max_tokens: int = 500
    
    def append(self, text: str) -> bool:
        new_content = self.content + "\n" + text
        if len(new_content) > self.max_tokens * 4:  # rough char estimate
            return False  # would overflow
        self.content = new_content.strip()
        return True
    
    def replace(self, old: str, new: str) -> bool:
        if old not in self.content:
            return False
        self.content = self.content.replace(old, new, 1)
        return True

class ArchivalMemory:
    """VectorDB-backed unlimited archival memory."""
    def __init__(self):
        self.entries: List[Dict] = []
    
    def insert(self, content: str, metadata: Dict = None) -> str:
        entry = {
            "id": len(self.entries),
            "content": content,
            "metadata": metadata or {},
            "timestamp": time.time(),
        }
        self.entries.append(entry)
        return f"Inserted entry #{entry['id']}"
    
    def search(self, query: str, n: int = 3) -> List[Dict]:
        """Simple keyword search (production: use embeddings)."""
        query_words = set(query.lower().split())
        scored = []
        for entry in self.entries:
            overlap = len(query_words & set(entry["content"].lower().split()))
            scored.append((overlap, entry))
        scored.sort(key=lambda x: -x[0])
        return [e for _, e in scored[:n] if _ > 0]

class RecallMemory:
    """Searchable conversation history."""
    def __init__(self, max_recent: int = 20):
        self.history: deque = deque(maxlen=max_recent)
    
    def add(self, role: str, content: str):
        self.history.append({"role": role, "content": content, "timestamp": time.time()})
    
    def search(self, query: str, n: int = 5) -> List[Dict]:
        query_words = set(query.lower().split())
        scored = [(sum(w in m["content"].lower() for w in query_words), m) 
                  for m in self.history]
        return [m for _, m in sorted(scored, key=lambda x: -x[0])[:n] if _ > 0]

# --- MemGPT Agent ---
class MemGPTAgent:
    def __init__(self, persona: str = "Ada", human_name: str = "User"):
        # Core memory blocks
        self.core_memory = {
            "persona": CoreMemoryBlock("persona", f"I am {persona}, an AI assistant with persistent memory."),
            "human": CoreMemoryBlock("human", f"The user's name is {human_name}."),
        }
        # External storage
        self.archival = ArchivalMemory()
        self.recall = RecallMemory()
        
        # Internal monologue (visible to us, not user)
        self.inner_thoughts: List[str] = []
        
        print(f"[MemGPT] Agent '{persona}' initialized with virtual context management.")
    
    # --- Memory Tools (callable by the LLM) ---
    def core_memory_append(self, block_name: str, content: str) -> str:
        if block_name not in self.core_memory:
            return f"Error: block '{block_name}' not found"
        success = self.core_memory[block_name].append(content)
        return f"Appended to {block_name}" if success else f"Error: {block_name} block full"
    
    def core_memory_replace(self, block_name: str, old_content: str, new_content: str) -> str:
        if block_name not in self.core_memory:
            return f"Error: block '{block_name}' not found"
        success = self.core_memory[block_name].replace(old_content, new_content)
        return f"Updated {block_name}" if success else f"Error: '{old_content}' not found in {block_name}"
    
    def archival_memory_insert(self, content: str) -> str:
        result = self.archival.insert(content)
        return result
    
    def archival_memory_search(self, query: str, page: int = 0) -> str:
        results = self.archival.search(query)
        if not results:
            return "No results found in archival memory."
        return "\n".join(f"[{r['id']}] {r['content'][:200]}" for r in results)
    
    def conversation_search(self, query: str) -> str:
        results = self.recall.search(query)
        if not results:
            return "No matching conversation history."
        return "\n".join(f"[{r['role']}] {r['content'][:150]}" for r in results)
    
    def send_message(self, content: str) -> str:
        """The ONLY way the agent communicates with the user."""
        return content
    
    # --- Simulated Agent Step ---
    def step(self, user_message: str) -> str:
        """Process one user message through the MemGPT loop."""
        print(f"\n{'─'*50}")
        print(f"👤 User: {user_message}")
        
        # 1. Store in recall memory
        self.recall.add("user", user_message)
        
        # 2. Simulate LLM deciding what memory ops to run
        tool_calls, response = self._mock_agent_decision(user_message)
        
        # 3. Execute memory tool calls
        for call in tool_calls:
            tool_name = call["tool"]
            args = call["args"]
            tool_fn = getattr(self, tool_name, None)
            if tool_fn:
                result = tool_fn(**args)
                self.inner_thoughts.append(f"[{tool_name}] → {result}")
                print(f"  🧠 Memory op: {tool_name}({list(args.values())[0][:40] if args else ''}...r)")
                print(f"     Result: {result}")
        
        # 4. Store response in recall
        self.recall.add("assistant", response)
        print(f"🤖 Agent: {response}")
        
        return response
    
    def _mock_agent_decision(self, message: str):
        """Simulate the LLM deciding on memory operations + response."""
        tool_calls = []
        
        msg_lower = message.lower()
        
        # Remember new facts about the user
        if "my name is" in msg_lower:
            name = message.split("my name is")[-1].strip().split()[0].rstrip(".,!")
            tool_calls.append({
                "tool": "core_memory_replace",
                "args": {"block_name": "human", "old_content": "The user's name is User.", "new_content": f"The user's name is {name}."}
            })
        
        # Store important info in archival
        if any(w in msg_lower for w in ["remember", "note that", "important:", "don't forget"]):
            tool_calls.append({
                "tool": "archival_memory_insert",
                "args": {"content": f"User said: {message}"}
            })
        
        # Search archival for relevant context
        if any(w in msg_lower for w in ["what did", "recall", "earlier", "before", "previous"]):
            tool_calls.append({
                "tool": "archival_memory_search",
                "args": {"query": message}
            })
            tool_calls.append({
                "tool": "conversation_search",
                "args": {"query": message}
            })
        
        # Generate response
        human_name = self.core_memory["human"].content.split("name is")[-1].strip().rstrip(".")
        persona_name = self.core_memory["persona"].content.split("I am")[-1].split(",")[0].strip()
        
        if "recall" in msg_lower or "what did" in msg_lower:
            response = f"I'm searching my memory... {self.archival_memory_search(message)}"
        elif "my name is" in msg_lower:
            name = message.split("my name is")[-1].strip().split()[0].rstrip(".,!")
            response = f"Nice to meet you, {name}! I've updated my memory with your name and will remember it in future conversations."
        else:
            response = f"I understand your message, {human_name}. {message[:50]}... I've processed this in my context."
        
        return tool_calls, response
    
    def show_memory_state(self):
        print(f"\n{'='*50}")
        print("📊 MEMORY STATE")
        print(f"{'='*50}")
        for name, block in self.core_memory.items():
            print(f"Core [{name}]: {block.content[:80]}")
        print(f"Archival entries: {len(self.archival.entries)}")
        print(f"Recall entries: {len(self.recall.history)}")
        if self.archival.entries:
            print("Recent archival:", self.archival.entries[-1]["content"][:80])

# --- Demo ---
agent = MemGPTAgent(persona="Ada", human_name="User")

# Simulate a conversation
agent.step("Hello! My name is Sarah.")
agent.step("Remember that I'm working on a thesis about climate change.")
agent.step("I prefer detailed explanations with examples.")
agent.step("What did I tell you to remember earlier?")
agent.step("Can you recall what my research topic is?")

agent.show_memory_state()

[MemGPT] Agent 'Ada' initialized with virtual context management.

──────────────────────────────────────────────────
👤 User: Hello! My name is Sarah.
  🧠 Memory op: core_memory_replace(human...r)
     Result: Updated human
🤖 Agent: Nice to meet you, Hello! I've updated my memory with your name and will remember it in future conversations.

──────────────────────────────────────────────────
👤 User: Remember that I'm working on a thesis about climate change.
  🧠 Memory op: archival_memory_insert(User said: Remember that I'm working on ...r)
     Result: Inserted entry #0
🤖 Agent: I understand your message, Hello. Remember that I'm working on a thesis about climat... I've processed this in my context.

──────────────────────────────────────────────────
👤 User: I prefer detailed explanations with examples.
🤖 Agent: I understand your message, Hello. I prefer detailed explanations with examples.... I've processed this in my context.

──────────────────────────────────────────────────
👤 User

## 6. SuperAGI Production Agent Framework

**SuperAGI** is an open-source production framework for building, managing, and running autonomous agents at scale.

### Key Features

| Feature | Description |
|---------|-------------|
| **GUI Dashboard** | Web UI for creating, running, monitoring agents |
| **Concurrent Agents** | Run multiple agents in parallel with resource management |
| **Agent Marketplace** | Pre-built agent templates (marketing, coding, research) |
| **Tool Marketplace** | 50+ tools: GitHub, Jira, Slack, Email, Browser, Code |
| **Memory** | Built-in vector memory (Pinecone, Weaviate, Redis) |
| **Telemetry** | Agent run logs, performance metrics, token tracking |
| **Permissions** | Human-in-the-loop approval gates |

### Architecture

```
┌─────────────────────────────────────────────┐
│              SuperAGI Platform               │
│                                              │
│  ┌──────────┐    ┌──────────┐               │
│  │ Agent 1  │    │ Agent 2  │  ← concurrent │
│  │ (Coder)  │    │(Marketer)│               │
│  └────┬─────┘    └────┬─────┘               │
│       │               │                     │
│       └───────┬────────┘                    │
│               │                             │
│  ┌────────────▼───────────────────────┐     │
│  │         Tool Registry              │     │
│  │  GitHub | Browser | Email | Slack  │     │
│  │  Code Executor | File | Searcher   │     │
│  └────────────────────────────────────┘     │
│                                              │
│  ┌────────────────────────────────────┐     │
│  │         Vector Memory              │     │
│  │    Pinecone / Weaviate / Redis     │     │
│  └────────────────────────────────────┘     │
└─────────────────────────────────────────────┘
```

### Installation & Quick Start (real commands)

```bash
git clone https://github.com/TransformerOptimus/SuperAGI.git
cd SuperAGI
cp config_template.yaml config.yaml
# Edit config.yaml with API keys
docker-compose up --build
# Access GUI at http://localhost:3000
```

SuperAGI is best suited for **production deployments** where you need monitoring, concurrent execution, and a user-friendly interface for non-technical users.

## 7. Phidata / Agno Workflow-Based Multi-Modal Agents

**Phidata** (rebranded as **Agno**) takes a **workflow-first** approach to agents. Instead of free-form loops, agents follow structured workflows with built-in storage, memory, and multi-modal support.

### Key Concepts

```python
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools.yfinance import YFinanceTools
from agno.storage.sqlite import SqliteStorage

# Agent with memory + tools
agent = Agent(
    model=OpenAIChat(id="gpt-4o"),
    tools=[DuckDuckGoTools(), YFinanceTools()],
    storage=SqliteStorage(table_name="agent_runs", db_file="agent.db"),
    add_history_to_messages=True,  # auto memory
    markdown=True,
)
```

### Team of Agents

```python
from agno.team import Team

web_agent = Agent(name="Web Agent", tools=[DuckDuckGoTools()])
finance_agent = Agent(name="Finance Agent", tools=[YFinanceTools()])

team = Team(
    name="Research Team",
    agents=[web_agent, finance_agent],
    mode="coordinate",  # or "collaborate", "route"
)
team.print_response("Compare NVIDIA vs AMD performance this quarter")
```

### Built-in Tools

| Category | Tools |
|----------|-------|
| **Search** | DuckDuckGo, Google, Tavily, Exa |
| **Finance** | YFinance, Alpha Vantage |
| **Research** | Arxiv, Wikipedia, PubMed |
| **Media** | YouTube, Reddit |
| **Code** | Python REPL, Shell |
| **Storage** | SQL, Files, S3 |

### Multi-Modal Support

Agno agents natively handle images, audio, and video alongside text passing them directly to multi-modal LLMs without boilerplate conversion code.

## 8. Composio Tool Integration Platform for Agents

**Composio** solves the **tool integration problem**: connecting agents to 100+ external services (GitHub, Slack, Gmail, Jira, Notion, etc.) with:

- **Managed OAuth/API key auth** no manual token handling
- **MCP compatibility** works as an MCP server
- **Framework adapters** LangChain, LlamaIndex, CrewAI, AutoGen, OpenAI SDK

### Architecture

```
┌─────────────────────────────────────────────────────┐
│                  Your Agent                          │
│  (LangChain / CrewAI / AutoGen / OpenAI Agents SDK) │
└──────────────────────┬──────────────────────────────┘
                       │ tool calls
                       ▼
┌─────────────────────────────────────────────────────┐
│                Composio Platform                     │
│                                                      │
│  ┌────────────┐  ┌────────────┐  ┌────────────┐    │
│  │    Auth    │  │   Action   │  │  Trigger   │    │
│  │  Manager  │  │  Registry  │  │  Manager   │    │
│  │ (OAuth2,  │  │(100+ apps, │  │(webhooks,  │    │
│  │  API key) │  │1000+actions│  │  polling)  │    │
│  └────────────┘  └────────────┘  └────────────┘    │
└──────────────────────┬──────────────────────────────┘
                       │ authenticated API calls
          ┌────────────┼───────────────┐
          ▼            ▼               ▼
     ┌────────┐   ┌────────┐    ┌────────────┐
     │ GitHub │   │  Slack │    │   Gmail    │
     └────────┘   └────────┘    └────────────┘
```

### Quick Integration

```python
from composio_openai import ComposioToolSet, App, Action
from openai import OpenAI

toolset = ComposioToolSet()
tools = toolset.get_tools(actions=[
    Action.GITHUB_CREATE_ISSUE,
    Action.SLACK_SEND_MESSAGE,
    Action.GMAIL_SEND_EMAIL,
])

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Create a GitHub issue for login bug"}],
    tools=tools,
)
# Composio handles auth + execution automatically
result = toolset.handle_tool_calls(response)
```

### MCP Integration

```bash
# Start Composio as MCP server
composio mcp start --apps github,slack,gmail

# Claude Code or any MCP client can now use these tools
```

## 9. Agent Comparison Matrix

| Framework | Autonomy | Memory | Tool Access | Multi-Agent | Production Ready | Best For |
|-----------|----------|--------|-------------|-------------|-----------------|----------|
| **AutoGPT** | Very High | Context + VectorDB | File, Web, Code | No | No | Experiments |
| **BabyAGI** | High | VectorDB | Minimal | No | No | Learning/Prototyping |
| **MetaGPT** | Medium | Role-local | Code, Files | Yes (roles) | Partial | Software teams |
| **OpenAI Agents SDK** | Medium | Configurable | Any (tools) | Yes | Yes | Production APIs |
| **MemGPT/Letta** | Medium | Hierarchical (infinite) | Memory ops | Yes | Yes | Long-context apps |
| **SuperAGI** | High | VectorDB | 50+ tools | Yes | Yes | GUI-based ops |
| **Phidata/Agno** | Medium | Built-in + storage | 20+ tool sets | Yes (Teams) | Yes | Workflow agents |
| **CrewAI** | Medium | Shared context | Via tools | Yes | Yes | Task pipelines |
| **LangGraph** | Low-High | Configurable | Via nodes | Yes | Yes | Complex workflows |

### Autonomy Levels

$$\text{Autonomy} = \frac{\text{Decisions made by agent}}{\text{Total decisions in workflow}}$$

- **Low** (0-30%): Human approves each major step
- **Medium** (30-70%): Agent handles sub-tasks, human reviews outputs
- **High** (70-90%): Agent runs independently, human reviews final output
- **Very High** (90-100%): Fully autonomous, minimal human oversight

## 10. Agent Failure Modes

Understanding how agents fail is as important as building them.

### 1. Infinite Loops

The agent repeats the same action because it lacks loop detection:

```
Step 1: Search for X → found partial info
Step 2: Search for X (again) → same partial info
Step 3: Search for X (again) → same partial info
... (forever)
```

**Fix**: Track action history; detect repeated (action, args) pairs; add max-steps limit.

### 2. Context Overflow

After many steps, the context fills up and early instructions are lost:

$$\text{Effective Instructions} \propto e^{-\frac{\text{steps}}{\text{context\_length}}}$$

**Fix**: MemGPT-style memory management; summarize old turns; use hierarchical context.

### 3. Hallucinated Tool Calls

The agent invokes tools with fabricated arguments or invents tool names:

```json
{"tool": "check_database", "args": {"query": "SELECT * FROM users WHERE id=1"}}
// But "check_database" tool doesn't exist!
```

**Fix**: Strict tool schema validation; graceful error handling; re-prompt on failure.

### 4. Reward Hacking

The agent finds shortcuts that satisfy the stated metric but not the true goal:

| Stated Goal | Agent Behavior | Actual Outcome |
|-------------|----------------|----------------|
| "Maximize test pass rate" | Delete failing tests | 100% pass rate, broken code |
| "Minimize user complaints" | Block complaint submissions | Zero complaints, unhappy users |
| "Complete tasks quickly" | Mark tasks done without executing | Fast, zero actual work |

### 5. Goal Misgeneralization

The agent learned to solve a proxy task, not the real goal:

```
Training: "Summarize documents" (always English docs)
Deployment: "Summarize documents" (French doc)
Agent: Translates to English first, then summarizes
→ Correct behavior from wrong generalization
```

### 6. Tool Cascade Failures

One failed tool call corrupts the entire reasoning chain downstream.

**Mitigation strategies**:
- Circuit breakers on tools
- Retry with exponential backoff
- Fallback tools for each tool
- Human-in-the-loop checkpoints

## 11. Agent Evaluation Frameworks

Measuring agent performance requires specialized benchmarks.

### Benchmark Overview

| Benchmark | Task Type | Metric | Difficulty |
|-----------|-----------|--------|------------|
| **SWE-bench** | GitHub issue → code fix | % resolved | Hard |
| **WebArena** | Web browser tasks | Task completion rate | Hard |
| **τ-bench (tau-bench)** | Tool-use + reasoning | Pass rate | Medium-Hard |
| **GAIA** | General AI assistant | % correct | Medium |
| **AgentBench** | OS, DB, lateral thinking | Overall score | Hard |
| **AgentEval** | Multi-domain tasks | Critic LLM score | Configurable |
| **inspection_evals** | Safety evaluation | Pass/fail | Safety-focused |

### SWE-bench

SWE-bench tests whether agents can resolve **real GitHub issues** from 12 popular Python repos (Django, Flask, NumPy, etc.):

$$\text{Score} = \frac{\text{Issues where all tests pass after agent's patch}}{\text{Total issues}}$$

State of the art (2024): ~50% on SWE-bench Verified (Claude 3.5 Sonnet + SWE-agent)

### GAIA

GAIA (General AI Assistants) tests **real-world question answering** requiring multi-step reasoning and tool use across 3 levels of difficulty.

### AgentBench

AgentBench evaluates agents across 8 environments:
- Operating System (shell commands)
- Database (SQL queries)  
- Knowledge Graph
- Digital Card Game
- Lateral Thinking Puzzles
- House Holding (embodied)
- Web Shopping
- Web Browsing

### Evaluation Best Practices

```python
# AgentEval pattern: use LLM as judge
def evaluate_agent_run(task: str, agent_output: str, ground_truth: str) -> Dict:
    prompt = f"""
    Task: {task}
    Agent Output: {agent_output}
    Ground Truth: {ground_truth}
    
    Rate the agent output on:
    1. Correctness (0-10)
    2. Completeness (0-10)  
    3. Efficiency (0-10)
    
    Return JSON: {{"correctness": N, "completeness": N, "efficiency": N, "reasoning": "..."}}
    """
    # result = llm(prompt)
    # return json.loads(result)
    return {"correctness": 8, "completeness": 7, "efficiency": 9, "reasoning": "Good approach"}
```

## 12. Computer Use Agents

Computer use agents can interact with GUIs, browsers, and operating systems not just APIs.

### Claude Computer Use API

Anthropic's Claude 3.5 Sonnet supports **computer use** with three core tools:

| Tool | Action |
|------|--------|
| `computer` | Screenshot, mouse click/move, keyboard input |
| `text_editor` | View, create, edit files |
| `bash` | Execute shell commands |

```python
import anthropic

client = anthropic.Anthropic()

# Claude can take screenshots, click, type
response = client.beta.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    tools=[
        {"type": "computer_20241022", "name": "computer", 
         "display_width_px": 1920, "display_height_px": 1080},
        {"type": "text_editor_20241022", "name": "str_replace_editor"},
        {"type": "bash_20241022", "name": "bash"},
    ],
    messages=[{"role": "user", "content": "Take a screenshot and describe the desktop"}],
    betas=["computer-use-2024-10-22"],
)
```

### SWE-agent

SWE-agent wraps LLMs with a specialized **Agent-Computer Interface (ACI)**:

- Custom file viewing (syntax highlighted, line numbers)
- Context-aware editing (no full file rewrite needed)
- Linting before saving
- Search/navigation tools

### OpenHands (formerly OpenDevin)

OpenHands is an open-source platform for software development agents:
- **Sandboxed Docker execution** safe code running
- **Multi-agent** specialized sub-agents for different tasks
- **Browser integration** Playwright-based web interaction
- **Eval framework** built-in SWE-bench evaluation

### Playwright Automation Agents

```python
# Agent + Playwright for browser automation
from playwright.async_api import async_playwright

async def browser_agent_step(action: dict) -> str:
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        
        if action["type"] == "navigate":
            await page.goto(action["url"])
        elif action["type"] == "click":
            await page.click(action["selector"])
        elif action["type"] == "type":
            await page.fill(action["selector"], action["text"])
        elif action["type"] == "screenshot":
            screenshot = await page.screenshot()
            return screenshot  # send to vision model
        
        await browser.close()
        return "Action completed"
```

### Key Challenge: Grounding

The core challenge is **visual grounding** accurately mapping natural language actions ("click the submit button") to pixel coordinates or DOM selectors. Models must reason about:
- UI layout and element positions
- State changes after each action
- Error recovery when UI is unexpected

## 13. Voice Agents

Voice agents combine STT (speech-to-text), LLM, and TTS (text-to-speech) into real-time conversational AI. The key challenge is **latency** humans expect responses in under 500ms.

### Latency Budget

$$\text{Total Latency} = \underbrace{T_{STT}}_{\text{50-200ms}} + \underbrace{T_{LLM}}_{\text{200-500ms}} + \underbrace{T_{TTS}}_{\text{50-200ms}} + \underbrace{T_{network}}_{\text{10-50ms}}$$

**Target**: $T_{total} < 500\text{ms}$ for natural conversation

### Platforms Comparison

| Platform | Approach | Latency | Key Feature |
|----------|----------|---------|-------------|
| **Vapi** | Full-stack voice API | ~300ms | Phone calls, web, SIP |
| **Retell AI** | Turn-by-turn voice | ~200ms | Interruption handling |
| **OpenAI Realtime API** | WebSocket, server VAD | ~150ms | Native audio tokens |
| **ElevenLabs Conv. AI** | End-to-end | ~250ms | Ultra-realistic TTS |
| **Deepgram Aura** | TTS only | ~80ms | Lowest TTS latency |

### OpenAI Realtime API (WebSocket)

```javascript
// WebSocket-based real-time audio streaming
const ws = new WebSocket("wss://api.openai.com/v1/realtime?model=gpt-4o-realtime-preview");

ws.onopen = () => {
    // Configure session
    ws.send(JSON.stringify({
        type: "session.update",
        session: {
            voice: "alloy",
            instructions: "You are a helpful assistant. Be concise.",
            input_audio_transcription: { model: "whisper-1" },
            turn_detection: { type: "server_vad", threshold: 0.5 },
        }
    }));
};

ws.onmessage = (event) => {
    const msg = JSON.parse(event.data);
    if (msg.type === "response.audio.delta") {
        // Stream audio chunk to speakers
        playAudioChunk(msg.delta);
    }
};
```

### Latency Optimization Techniques

1. **Streaming TTS**: Start speaking before full LLM response
2. **End-of-utterance detection**: VAD (Voice Activity Detection) to cut off quickly
3. **Interruption handling**: Detect when user interrupts, cancel LLM/TTS
4. **Warm caching**: Keep WebSocket connections alive
5. **Edge deployment**: Run STT/TTS on edge nodes near users
6. **Response length control**: Instruct LLM to give short answers
7. **Speculative execution**: Start TTS on partial LLM output

### Architecture for Sub-500ms Voice Agent

```
User Speech → [Deepgram STT, streaming] → partial transcript
                                                │
                    [VAD detects end of speech] │
                                                ▼
                              [GPT-4o Realtime / streaming]
                                                │
                                    first tokens arrive
                                                ▼
                              [ElevenLabs / Cartesia TTS]
                                    streaming audio
                                                │
                                                ▼
                                     User hears response
                               (while LLM is still generating)
```

## Additional Learning Resources

### Papers

| Paper | Description | Link |
|-------|-------------|------|
| **MetaGPT** (Hong et al., 2023) | Multi-agent software development | arxiv.org/abs/2308.00352 |
| **BabyAGI** (Nakajima, 2023) | Task-driven autonomous agent | github.com/yoheinakajima/babyagi |
| **MemGPT** (Packer et al., 2023) | LLMs as OS with memory management | arxiv.org/abs/2310.08560 |
| **AgentBench** (Liu et al., 2023) | Benchmarking LLMs as agents | arxiv.org/abs/2308.03688 |
| **WebArena** (Zhou et al., 2023) | Realistic web agent benchmark | arxiv.org/abs/2307.13854 |
| **SWE-bench** (Jimenez et al., 2023) | Software engineering benchmark | arxiv.org/abs/2310.06770 |
| **GAIA** (Mialon et al., 2023) | General AI assistant benchmark | arxiv.org/abs/2311.12983 |
| **Cognitive Architectures for Language Agents** (Sumers et al., 2024) | Survey of agent architectures | arxiv.org/abs/2309.02427 |

### Official Documentation

- **OpenAI Agents SDK**: platform.openai.com/docs/guides/agents
- **MetaGPT**: docs.deepwisdom.ai
- **Letta (MemGPT)**: docs.letta.com
- **Phidata/Agno**: docs.agno.com
- **Composio**: docs.composio.dev
- **SuperAGI**: superagi.com/docs
- **OpenHands**: docs.all-hands.dev
- **Claude Computer Use**: docs.anthropic.com/claude/docs/computer-use

### Courses & Tutorials

- **DeepLearning.AI Multi AI Agent Systems with crewAI** (Andrew Ng)
- **DeepLearning.AI AI Agents in LangGraph**
- **DeepLearning.AI Building Agentic RAG with LlamaIndex**
- **DeepLearning.AI Practical Multi AI Agents and Advanced Use Cases**
- **Hugging Face Agents Course** (free, huggingface.co/learn/agents-course)

### Key GitHub Repos

```bash
# MetaGPT
git clone https://github.com/geekan/MetaGPT

# OpenAI Agents SDK  
pip install openai-agents

# BabyAGI original
git clone https://github.com/yoheinakajima/babyagi

# Letta (MemGPT)
pip install letta

# SuperAGI
git clone https://github.com/TransformerOptimus/SuperAGI

# Phidata/Agno
pip install agno

# Composio
pip install composio-openai

# OpenHands
git clone https://github.com/All-Hands-AI/OpenHands

# SWE-agent
git clone https://github.com/SWE-agent/SWE-agent
```